# De un vídeo a un paquete certificado

Este cuaderno hace **lo único que no se puede hacer en las máquinas de casa**: la
reconstrucción densa, que exige CUDA. Todo lo demás —auditar, medir, certificar—
corre igual de bien en un portátil sin GPU, y por eso se queda fuera de aquí.

Lo que sale al final no es un `.obj` suelto: es un **paquete de reconstrucción**
sellado, con su manifest, sus hashes y sus cámaras, que la cadena de SoftSight
sabe leer. Antes de descargarlo lo inspecciona aquí mismo, así que sabes si vale
**antes** de bajarte 300 MB.

Orden de las celdas, y ninguna se salta:

```text
1  GPU            sin CUDA no hay densa, y más vale saberlo en el segundo cero
2  COLMAP         compilado con CUDA; se cachea en Drive para no repetirlo
3  fotogramas     el vídeo a JPEG, con el filtro de nitidez
4  disperso       cámaras y nube: de dónde se miró
5  densa + malla  lo que necesita la tarjeta
6  paquete        cámaras + nube + malla, sellado
7  veredicto      COMPLETE/PASS aquí, antes de descargar
8  descarga
```


## 1 — La tarjeta

Si esto no imprime una GPU, **para**: en `Entorno de ejecución → Cambiar tipo de
entorno` hay que elegir T4. Sin ella la celda 5 falla media hora más tarde, que es
la peor forma de enterarse.


In [ ]:
import subprocess, sys

salida = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if salida.returncode != 0:
    sys.exit('sin GPU: Entorno de ejecucion -> Cambiar tipo de entorno -> T4')
print(salida.stdout.split('\n')[8])
print(salida.stdout.split('\n')[9])


## 2 — COLMAP con CUDA

**El paquete de `apt` no sirve.** Está compilado sin CUDA, y `patch_match_stereo`
contesta *«Dense stereo reconstruction requires CUDA»* aunque la tarjeta esté ahí.
Hay que compilarlo, y son entre 25 y 40 minutos.

Una vez. La celda guarda el resultado en tu Drive y las siguientes sesiones lo
recuperan en segundos: Colab borra el disco al desconectar, Drive no.


In [ ]:
from pathlib import Path
import os, subprocess

USAR_DRIVE = True  # False = compilar cada sesion
CACHE = Path('/content/drive/MyDrive/colmap-cuda.tar.gz')

if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

def sh(orden, **kw):
    print('$', orden)
    subprocess.run(orden, shell=True, check=True, **kw)

if USAR_DRIVE and CACHE.exists():
    sh(f'tar xzf {CACHE} -C /usr/local')
    print('COLMAP recuperado de Drive')
else:
    sh('apt-get update -qq')
    sh('apt-get install -y -qq git cmake ninja-build build-essential '
       'libboost-program-options-dev libboost-graph-dev libboost-system-dev '
       'libeigen3-dev libflann-dev libfreeimage-dev libmetis-dev libgoogle-glog-dev '
       'libgtest-dev libgmock-dev libsqlite3-dev libglew-dev qtbase5-dev '
       'libqt5opengl5-dev libcgal-dev libceres-dev libcurl4-openssl-dev')
    sh('git clone --depth 1 https://github.com/colmap/colmap.git /content/colmap')
    # 75 = Turing, que es la T4. Compilar para una sola arquitectura ahorra
    # la mitad del tiempo; si Colab te da otra tarjeta, cambia el numero.
    sh('cmake -S /content/colmap -B /content/colmap/build -GNinja '
       '-DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=75 '
       '-DCMAKE_INSTALL_PREFIX=/usr/local')
    sh('ninja -C /content/colmap/build install')
    if USAR_DRIVE:
        sh('tar czf /content/colmap-cuda.tar.gz -C /usr/local bin/colmap lib share/colmap')
        sh(f'cp /content/colmap-cuda.tar.gz {CACHE}')

sh('colmap -h | head -3')


## 3 — El vídeo a fotogramas

Tres decisiones, y las tres deciden más que cualquier parámetro de COLMAP:

**Cuántos.** Entre 100 y 300 para un objeto. Más no mejora: el emparejamiento
crece con el cuadrado y llenas la sesión sin ver más superficie.

**Cuáles.** El filtro se queda con el fotograma **más nítido** de cada tramo. Un
vídeo a mano tiene desenfoque de movimiento en la mitad de los fotogramas, y uno
movido no solo no aporta: mete puntos falsos que arrastran la pose.

**En JPEG.** El productor lee las dimensiones de la cabecera JPEG, sin
descomprimir. Un PNG lo rechaza.


In [ ]:
VIDEO = '/content/entrada.mp4'   # sube el tuyo aqui
OBJETIVO = 200                    # fotogramas que quieres

from google.colab import files
if not Path(VIDEO).exists():
    subido = files.upload()
    Path(list(subido)[0]).rename(VIDEO)

sh('apt-get install -y -qq ffmpeg')

def sondear(campo, flujo='v:0'):
    return subprocess.run(
        f'ffprobe -v error -select_streams {flujo} -show_entries stream={campo} '
        f'-of csv=p=0 {VIDEO}', shell=True, capture_output=True, text=True).stdout.strip()

num, den = sondear('r_frame_rate').split('/')
fps = float(num) / float(den)
total = round(fps * float(subprocess.run(
    f'ffprobe -v error -show_entries format=duration -of csv=p=0 {VIDEO}',
    shell=True, capture_output=True, text=True).stdout.strip()))

# `thumbnail=N` se queda con el mas nitido de cada N, asi que N sale de lo que
# hay dividido entre lo que quieres. Calcularlo con 30 fps fijos daria de mas o
# de menos en cuanto el video no fuera de 30.
tramo = max(2, total // OBJETIVO)
Path('/content/frames').mkdir(exist_ok=True)
sh(f'ffmpeg -loglevel error -i {VIDEO} -vf thumbnail={tramo} -vsync 0 -q:v 2 '
   '/content/frames/f%04d.jpg')

marcos = sorted(Path('/content/frames').glob('*.jpg'))
print(f'{len(marcos)} fotogramas de {total} ({fps:.1f} fps), uno por cada {tramo}')
assert len(marcos) >= 20, 'con menos de 20 vistas no hay reconstruccion que valga'


## 4 — El disperso: dónde estaba la cámara

`sequential_matcher` y no `exhaustive`, porque esto viene de un vídeo: los
fotogramas consecutivos **ya sabes** que se solapan, y comparar todos contra todos
cuesta el cuadrado para descubrir lo que el orden ya te decía.

Si al final hay deriva —el final no cierra con el principio— es que falta cierre
de bucle: ahí sí compensa `--SequentialMatching.loop_detection 1`.


In [ ]:
W = Path('/content/trabajo'); W.mkdir(exist_ok=True)

sh(f'colmap feature_extractor --database_path {W}/db.db --image_path /content/frames '
   '--ImageReader.single_camera 1 --ImageReader.camera_model SIMPLE_RADIAL '
   '--SiftExtraction.use_gpu 1')
sh(f'colmap sequential_matcher --database_path {W}/db.db --SiftMatching.use_gpu 1')
(W/'sparse').mkdir(exist_ok=True)
sh(f'colmap mapper --database_path {W}/db.db --image_path /content/frames '
   f'--output_path {W}/sparse')

# En texto, que es lo que el productor lee. El binario tambien vale para COLMAP,
# pero el adaptador se escribio contra los .txt de la documentacion.
(W/'modelo').mkdir(exist_ok=True)
sh(f'colmap model_converter --input_path {W}/sparse/0 --output_path {W}/modelo '
   '--output_type TXT')

registradas = sum(1 for l in open(W/'modelo/images.txt') if l.strip() and not l.startswith('#'))//2
print(f'{registradas} de {len(marcos)} vistas registradas')
if registradas < len(marcos) * 0.6:
    print('AVISO: se perdio mas del 40 %. Suele ser desenfoque, o una vuelta demasiado rapida.')


## 5 — La densa y la malla

Esta es la celda que justifica Colab. `patch_match_stereo` es lo que no corre sin
CUDA, y es de lejos lo más caro: cuenta con 20–60 minutos según cuántas vistas.

**Delaunay y no Poisson, por defecto.** Poisson cierra los agujeros por
construcción, así que devuelve una superficie cerrada y bonita en la que **parte
de lo que ves no lo vio ninguna cámara**. Eso es interpolación, y el paquete
tendría que declararse `purelyReconstructed: false`. Delaunay deja el agujero
donde no hubo evidencia, que es más feo y más cierto.

Si quieres la cerrada, cambia `MALLADOR` y la celda 6 lo declarará sola.


In [ ]:
MALLADOR = 'delaunay'   # 'delaunay' = solo lo observado | 'poisson' = cierra agujeros

sh(f'colmap image_undistorter --image_path /content/frames --input_path {W}/sparse/0 '
   f'--output_path {W}/denso --output_type COLMAP')
sh(f'colmap patch_match_stereo --workspace_path {W}/denso --workspace_format COLMAP '
   '--PatchMatchStereo.geom_consistency true')
sh(f'colmap stereo_fusion --workspace_path {W}/denso --workspace_format COLMAP '
   f'--input_type geometric --output_path {W}/denso/fusion.ply')

if MALLADOR == 'poisson':
    sh(f'colmap poisson_mesher --input_path {W}/denso/fusion.ply --output_path {W}/malla.ply')
else:
    sh(f'colmap delaunay_mesher --input_path {W}/denso --output_path {W}/malla.ply')

print('malla:', (W/'malla.ply').stat().st_size // 1024, 'KiB')


## 6 — El paquete

Aquí entra SoftSight, y **solo para esto**: clona el repositorio y usa su
productor, que convierte la salida de COLMAP en un paquete sellado.

Las imágenes del paquete son las **originales**, no las rectificadas: las cámaras
declaran su distorsión y `imageSpace: ORIGINAL`, así que el paquete dice lo que
COLMAP calibró en vez de una versión ya corregida que nadie puede comprobar.


In [ ]:
# Node, en la version que SoftSight fija en su .nvmrc. El de Colab suele ser mas
# viejo y `npm ci` falla por `engines`, que es un fallo confuso de leer.
sh('git clone --depth 1 https://github.com/JesusGalindez/softsight.git /content/softsight')
nvmrc = Path('/content/softsight/.nvmrc').read_text().strip()
mayor = nvmrc.split('.')[0]
sh(f'curl -fsSL https://deb.nodesource.com/setup_{mayor}.x | bash -')
sh('apt-get install -y -qq nodejs')
sh('node --version')

sh('cd /content/softsight && npm ci --silent && npm run build:agent3d')

# El productor espera cameras/images/points3D y un images/ al lado.
sh(f'cp -r /content/frames {W}/modelo/images')

interpolada = '' if MALLADOR == 'poisson' else '--sin-interpolar'
sh(f'node /content/softsight/producers/colmap/build.mjs {W}/modelo /content/paquete '
   f'--vistas todas --malla {W}/malla.ply --id video-v1 {interpolada}')


## 7 — El veredicto, antes de descargar

Los dos ejes van por separado y significan cosas distintas: `execution` dice si se
pudo mirar, `certification` si lo que se vio cumple. Un paquete roto sale
`COMPLETE + FAIL`; uno al que le falta evidencia, `INCONCLUSIVE`.

Los avisos de cobertura no son errores: dicen **qué partes no vio ninguna cámara**,
que es justamente lo que un escaneo nunca te cuenta.


In [ ]:
import json

r = subprocess.run(['node', '/content/softsight/tools/reconstruction.mjs', 'inspect',
                    '/content/paquete/manifest.json'], capture_output=True, text=True)
informe = json.loads(r.stdout)
print(informe['execution'], '+', informe['certification'], ' (salida', r.returncode, ')')
for m in informe.get('measurements', []):
    print(f"  {m.get('name','')}: {m.get('value')}")
for a in informe.get('warnings', []):
    print(f"  aviso {a.get('code')}: {a.get('message','')}")

Path('/content/informe.json').write_text(r.stdout)


## 8 — Descargar

El paquete entero, con su informe al lado. En tu máquina se vuelve a inspeccionar
—los hashes del manifest comprueban que llegó igual que salió— y ahí ya no hace
falta GPU para nada.


In [ ]:
sh('cp /content/informe.json /content/paquete/informe.json')
sh('cd /content && zip -qr paquete.zip paquete')
print('tamano:', Path('/content/paquete.zip').stat().st_size // 1024 // 1024, 'MiB')
files.download('/content/paquete.zip')
